# DPO Training Launcher

Colab 런타임에서 `scripts/train_dpo.py`를 실행하기 위한 얇은 런처 노트북입니다.

GitHub repo와 실험 설정입니다. Colab에서 실행하기 전에 이 셀의 값만 수정하세요.

In [ ]:
GITHUB_REPO_URL = "https://github.com/c-peace/LEET_LLM_PEFT.git"
GITHUB_BRANCH = "main"
PROJECT_DIR = "/content/seonji-llm"

HF_USERNAME = "choipeace"
PROJECT_NAME = "LEET_seonji"
EXPERIMENT_VERSION = "aug-dpo-v1"
EXPERIMENT_TITLE = "augmented-dpo-clean-all"
EXPERIMENT_HYPOTHESIS = """SFT adapter에 DPO preference 학습을 추가하면 rejected 선지 패턴을 낮추고 선택지 품질이 개선되는지 확인한다."""
HF_REPO_PRIVATE = True

OUTPUT_ROOT = "/content/dpo_outputs/runs"
BASE_CONFIG_PATH = "configs/train_dpo_config_v1.json"
TMP_CONFIG_PATH = "/content/train_dpo_config_no_drive.json"
CLEAN_DPO_PATH = "dpo_train_clean_v1.json"
REVIEWED_DPO_PATH = "dpo_candidates_reviewed_v1.json"
CLEAN_SEED = 42

SFT_ADAPTERS = {
    "Qwen/Qwen3.5-4B": "choipeace/leet-seonji-qwen35-4b-aug-v1",
    "google/gemma-4-E4B-it": "choipeace/leet-seonji-gemma4-e4b-it-aug-v1",
    "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct": "choipeace/leet-seonji-exaone35-24b-instruct-aug-v1",
}


GitHub에서 프로젝트 코드를 가져오고 작업 폴더로 이동합니다.

In [ ]:
import os
import subprocess
from pathlib import Path

if "YOUR_GITHUB_ID" in GITHUB_REPO_URL:
    raise ValueError("GITHUB_REPO_URL을 본인 GitHub repo 주소로 바꾼 뒤 실행하세요.")

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print(f"cwd={Path.cwd()}")
required_paths = [
    "scripts/train_dpo.py",
    "scripts/upload_hf_adapter.py",
    "dpo_candidates_v1.json",
    "test_set_v1.json",
    "configs/train_dpo_config_v1.json",
    "configs/prompt_template_v1.md",
]
for required_path in required_paths:
    path = Path(required_path)
    if not path.exists():
        raise FileNotFoundError(f"required file not found: {path}")
    print(f"found: {path}")


런타임 GPU를 확인합니다.

In [ ]:
!nvidia-smi

필요 패키지를 설치합니다.

In [ ]:
!pip install -U transformers datasets peft accelerate bitsandbytes huggingface_hub trl

Hugging Face Hub에 로그인합니다. 토큰은 노트북 입력창에 직접 붙여넣고, 코드나 채팅에 남기지 마세요.

In [ ]:
from huggingface_hub import notebook_login, whoami

notebook_login()

user = whoami()
print(f"Logged in to Hugging Face as: {user['name']}")

DPO 후보에서 명확한 불량 케이스를 `quality_tier: exclude`로 표시하고, 나머지 전체를 학습용 clean 파일로 만듭니다.

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

source_path = Path("dpo_candidates_v1.json")
records = json.loads(source_path.read_text(encoding="utf-8"))

def normalize_choices(choices):
    if not isinstance(choices, list):
        return []
    return [str(choice).strip() for choice in choices]

def exclusion_reasons(record):
    reasons = []
    meta = record.get("meta", {})
    chosen = record.get("chosen", {})
    rejected = record.get("rejected", {})
    chosen_choices = normalize_choices(chosen.get("choices"))
    rejected_choices = normalize_choices(rejected.get("choices"))
    rejected_answer_index = rejected.get("answer_index")
    chosen_answer_index = chosen.get("answer_index")
    failure_type = meta.get("requested_failure_type")

    if len(rejected_choices) != 5:
        reasons.append("bad_rejected_choice_count")
    if len(chosen_choices) != 5:
        reasons.append("bad_chosen_choice_count")
    if any(not choice for choice in rejected_choices):
        reasons.append("empty_rejected_choice")
    if not isinstance(rejected_answer_index, int) or not 1 <= rejected_answer_index <= 5:
        reasons.append("bad_rejected_answer_index")
    if rejected_choices and rejected_choices == chosen_choices:
        reasons.append("rejected_choices_equal_chosen")
    if failure_type == "wrong_answer_index" and rejected_answer_index == chosen_answer_index:
        reasons.append("wrong_answer_index_same_as_chosen")
    if failure_type != "format_or_duplicate" and len(rejected_choices) == 5 and len(set(rejected_choices)) < 5:
        reasons.append("unexpected_duplicate_rejected_choices")
    return reasons

reviewed = []
reason_counts = Counter()
for record in records:
    item = dict(record)
    item["meta"] = dict(record.get("meta", {}))
    reasons = exclusion_reasons(item)
    if reasons:
        item["meta"]["quality_tier"] = "exclude"
        item["meta"]["exclude_reasons"] = reasons
        reason_counts.update(reasons)
    reviewed.append(item)

eligible = [item for item in reviewed if item.get("meta", {}).get("quality_tier") != "exclude"]
rng = random.Random(CLEAN_SEED)
rng.shuffle(eligible)
clean_train = eligible

Path(REVIEWED_DPO_PATH).write_text(json.dumps(reviewed, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
Path(CLEAN_DPO_PATH).write_text(json.dumps(clean_train, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"raw candidates: {len(records)}")
print(f"excluded: {len(reviewed) - len(eligible)}")
print(f"eligible: {len(eligible)}")
print(f"clean train: {len(clean_train)} -> {CLEAN_DPO_PATH}")
print(f"reviewed candidates: {REVIEWED_DPO_PATH}")
print(dict(reason_counts))


Drive 용량을 쓰지 않도록 학습 결과 저장 위치와 DPO 학습셋 경로를 Colab 임시 디스크 기준으로 바꿉니다.

In [ ]:
import json
from pathlib import Path

config_path = Path(BASE_CONFIG_PATH)
config = json.loads(config_path.read_text(encoding="utf-8"))
config["dataset"]["path"] = CLEAN_DPO_PATH
if config["dataset"].get("test_path") != "test_set_v1.json":
    raise ValueError(f"unexpected test dataset: {config['dataset'].get('test_path')}")
config["sft_adapters"] = SFT_ADAPTERS
config["outputs"]["root_dir"] = OUTPUT_ROOT
config["training"]["save_strategy"] = "no"

tmp_config_path = Path(TMP_CONFIG_PATH)
tmp_config_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"temporary config: {tmp_config_path}")
print(f"DPO dataset: {config['dataset']['path']}")
print(f"final test dataset: {config['dataset']['test_path']}")
print(f"output root: {config['outputs']['root_dir']}")
print(f"save_strategy: {config['training']['save_strategy']}")
print(f"experiment: {PROJECT_NAME}-{EXPERIMENT_VERSION} / {EXPERIMENT_TITLE}")


학습이 끝난 최신 run을 Hugging Face Hub에 업로드하는 helper 함수입니다. 각 모델 학습 셀 끝에서 자동 호출됩니다.

In [ ]:
import subprocess

def upload_latest_run():
    if HF_USERNAME == "YOUR_HF_USERNAME":
        raise ValueError("HF_USERNAME을 본인 Hugging Face 아이디로 바꾼 뒤 실행하세요.")

    subprocess.run(
        [
            "python",
            "scripts/upload_hf_adapter.py",
            "--hf_username",
            HF_USERNAME,
            "--project_name",
            PROJECT_NAME,
            "--experiment_version",
            EXPERIMENT_VERSION,
            "--experiment_title",
            EXPERIMENT_TITLE,
            "--experiment_hypothesis",
            EXPERIMENT_HYPOTHESIS,
            "--runs_root",
            OUTPUT_ROOT,
            "--private",
            str(HF_REPO_PRIVATE).lower(),
        ],
        check=True,
    )


Qwen DPO 학습 실행

In [ ]:
!python scripts/train_dpo.py \
  --config {TMP_CONFIG_PATH} \
  --model_name Qwen/Qwen3.5-4B \
  --sft_adapter {SFT_ADAPTERS['Qwen/Qwen3.5-4B']} \
  --copy_config

upload_latest_run()

# smoke test
# !python scripts/train_dpo.py \
#   --config {TMP_CONFIG_PATH} \
#   --model_name Qwen/Qwen3.5-4B \
#   --sft_adapter {SFT_ADAPTERS['Qwen/Qwen3.5-4B']} \
#   --run_id smoke_qwen_dpo \
#   --max_steps 10 \
#   --copy_config
# upload_latest_run()


Gemma DPO 학습 실행

In [ ]:
!python scripts/train_dpo.py \
  --config {TMP_CONFIG_PATH} \
  --model_name google/gemma-4-E4B-it \
  --sft_adapter {SFT_ADAPTERS['google/gemma-4-E4B-it']} \
  --copy_config

upload_latest_run()

EXAONE DPO 학습 실행

In [ ]:
!python scripts/train_dpo.py \
  --config {TMP_CONFIG_PATH} \
  --model_name LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct \
  --sft_adapter {SFT_ADAPTERS['LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct']} \
  --copy_config

upload_latest_run()

최근 결과 폴더와 평가 요약을 확인합니다.

In [ ]:
import json
from pathlib import Path

runs = sorted(Path(OUTPUT_ROOT).glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
for run in runs[:5]:
    print(run)

if runs:
    latest_run = runs[0]
    summary_path = latest_run / "eval_summary.json"
    results_path = latest_run / "eval_results.json"
    print()
    print(f"latest_run={latest_run}")
    if summary_path.exists():
        print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
    if results_path.exists():
        results = json.loads(results_path.read_text(encoding="utf-8"))
        print(f"dataset={results.get('dataset_path')}")
        print(f"test_dataset={results.get('eval_dataset_path')}")
        print(json.dumps(results.get("eval_summary", {}).get("test_generation_summary"), ensure_ascii=False, indent=2))


In [ ]:
from google.colab import runtime

runtime.unassign()
